# Combined ClinVar + UniProt RAG — query & summarize

Pipeline: **ingest both sources** → **index both** → **combined search** → **summarize**.

Prerequisites:
- `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`
- `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`

Single-source notebooks: `clinvar_rag.ipynb`, `uniprot_rag.ipynb`.

In [ ]:
# Natural-language query
query = "cellular apoptosis"

# "BRCA1 hereditary breast cancer pathogenic" → top 3 ClinVar variants (lowest distances)
# "CFTR cystic fibrosis" → mixed ranking: 2 ClinVar + 1 UniProt
# "mismatch repair mechanism" → top 3 UniProt variants 

In [ ]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [ ]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer

In [ ]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search_combined
from rag_summary import load_llm, summarize
from rag_export import append_rag_report

## Combined search

**Ranking:** Each source is queried independently (`TOP_K_CHUNKS` chunks per collection). Within each source, records are scored by their best (lowest-distance) chunk. Those per-record scores are merged and sorted globally; the top `TOP_K_RESULTS` records across ClinVar and UniProt are sent to the LLM (one full parquet row each).

In [ ]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool per collection; used for ranking (not sent to LLM)
TOP_K_RESULTS = 3  # top records globally across both sources

In [ ]:
# Paths, embedding model, per-source search configs
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]

UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]

SEARCH_CONFIGS = {
    "clinvar": {
        "collection_name": CLINVAR_COLLECTION,
        "parquet_path": CLINVAR_PARQUET,
        "group_key": "variation_id",
        "record_id_column": "VariationID",
        "record_id_meta_key": "variation_id",
        "record_fields": CLINVAR_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — variation_id={variation_id}, gene={gene}",
        "hit_fields": [("variation_id", "variation_id"), ("gene", "gene")],
        "full_record_label": "Full ClinVar record",
        "record_id_cast": int,
    },
    "uniprot": {
        "collection_name": UNIPROT_COLLECTION,
        "parquet_path": UNIPROT_PARQUET,
        "group_key": "entry_name",
        "record_id_column": "Entry",
        "record_id_meta_key": "accession",
        "record_fields": UNIPROT_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — entry_name={entry_name}, accession={accession}, gene={gene}",
        "hit_fields": [("entry_name", "entry_name"), ("accession", "accession"), ("gene", "gene")],
        "full_record_label": "Full UniProt record",
    },
}

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL)
result = search_combined(
    query,
    SEARCH_CONFIGS,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

## Summarize

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical genomics and protein biology assistant. "
    "Write a brief summary in plain English sentences that answers the user's query using ONLY the ClinVar and UniProt records provided. "
    "Clearly distinguish variant evidence (ClinVar: cite VariationID and Name) from protein evidence (UniProt: cite Entry Name and Protein Name). "
    "Do not invent genes, variants, proteins, significance, or other facts not present in the provided text. "
    "If information is missing, say so briefly."
)

Load summarizer → generate answer from retrieved context only.

- **local:** downloads once → cached under `~/.cache/huggingface/`; 0.5B ~2GB RAM.
- **openai:** calls `gpt-4o-mini` via API; set `OPENAI_API_KEY` in `.env`.

The LLM receives the top 3 records from the combined ranking (ClinVar and/or UniProt), not the raw chunk pool.

In [ ]:
# Summarization backend: "local" or "openai"
SUMMARIZER_BACKEND = "openai"
# Local model (SUMMARIZER_BACKEND == "local")
# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# OpenAI model (SUMMARIZER_BACKEND == "openai")
API_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 1024
REPORT_PATH = project_root / "reports/rag_log.md"
CSV_LOG_PATH = project_root / "data/reports/rag_log.csv"

In [ ]:
if SUMMARIZER_BACKEND == "local":
    llm_model, llm_tokenizer = load_llm(
        LLM_MODEL,
        model=globals().get("llm_model"),
        tokenizer=globals().get("llm_tokenizer"),
    )
else:
    llm_model, llm_tokenizer = None, None

In [ ]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    backend=SUMMARIZER_BACKEND,
    api_model=API_MODEL,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)

append_rag_report(
    REPORT_PATH,
    query,
    result,
    summary,
    search_mode="combined",
    summarizer_backend=SUMMARIZER_BACKEND,
    model_name=API_MODEL if SUMMARIZER_BACKEND == "openai" else LLM_MODEL,
    csv_path=CSV_LOG_PATH,
)